## Programação Orientada a Objetos em Python: Teoria e prática

A Programação Orientada a objetos, ou OPP/POO, organiza o programa em objetos que possuem.

* estado: dados armazenados nos atributos;
* comportamento: ações realizadas pelos métodos.

Antes dos desafios, precisamos fixar quatro conceitos básicos.

In [2]:
class Sensor:
    tipo = "genérico" # atributo de classe

    def __init__(self, identificador: str):
        self.identificador = identificador # atributo de instancia

    def exibir_identificador(self) -> str:
        return self.identificador

In [3]:
# Criando um objeto

sensor = Sensor("TEMP-001")

print(sensor.identificador)
print(sensor.exibir_identificador())

TEMP-001
TEMP-001


### Nesse código

| Elemento                 | Significado                       |
| ------------------------ | --------------------------------- |
| `Sensor`                 | Classe                            |
| `sensor`                 | Objeto ou instância               |
| `identificador`          | Atributo                          |
| `exibir_identificador()` | Método                            |
| `self`                   | Referência ao objeto atual        |
| `__init__`               | Inicializa os atributos do objeto |


# 1. Herança única

### Teoria

Herança permite criar uma classe nova aproveitando atributos e métodos de uma classe existente.

Na **herança única**, uma classe possui uma classe pai direita.

```bash
Equipamento
    ↓
Sensor
    ↓
SensorTemperatura
```

Nesse exemplo:

* `Sensor` herda diretamente de `Equipamento`;
* `SensorTemperatura` herda diretamente de `Sensor`;
* cada classe possui apenas um pai direto;

Essa estrutura também é chamada de **herança multinível**, pois existe uma sequência de classes.

A sintaxe é:


In [ ]:
class ClasseFilha(ClassePai):
    ...

A herança deve representar uma relação do tipo:
    
    -> A classe filha é um tipo de classe pai.

Por exemplo:
* sensor é um equipamento:
* sensor de temperatura é um sensor;
* modelo de regressão é um modelo de Machine Learning;


### Exemplo Prático
    

In [22]:
class Equipamento:
    def __init__(self, identificador: str, nome: str):
        self.identificador = identificador
        self.nome = nome
        self.ligado = False

    def ligar(self) -> None:
        self.ligado = True

    def desligar(self) -> None:
        self.ligado = False

    def exibir_status(self) -> str:
        status = "ligado" if self.ligado else "desligado"

        return (
            f"Identificador: {self.identificador}\n"
            f"Equipamento: {self.nome}\n"
            f"Status: {status}\n"
        )

A Classe `Equipamento` contém características genéricas que podem ser utilizadas por vários tipos de equipamentos.


Agora criamos a classe `Sensor`.

In [23]:
class Sensor(Equipamento):
    def __init__(
        self,
        identificador: str,
        nome: str,
        unidade: str
    ):
        super().__init__(identificador, nome)
        self.unidade = unidade

    def medir(self) -> float:
        raise NotImplementedError(
            "A classe filha deve implementar o método medir()."
        )

A linha:

```python
super().__init__(identificador, nome)
```

executa o `__init__` da classe anterior na hierarquia. Dessa forma, `Sensor` aproveita a inicialização de:

```python
self.identificador
self.nome
self.ligado
```

Depois, adiciona um novo atributo:

```python
self.unidade
```

Agora criamos uma classe especializada.

In [24]:
class SensorTemperatura(Sensor):
    def __init__(
        self,
        identificador: str,
        nome: str,
        temperatura: float
    ):
        super().__init__(
            identificador=identificador,
            nome=nome,
            unidade="°C"
        )

        self.temperatura = temperatura

    def medir(self) -> float:
        if not self.ligado:
            raise RuntimeError(
                "O sensor precisa estar ligado."
            )
        return self.temperatura

In [25]:
# Testando

sensor = SensorTemperatura(
    identificador="TEMP-001",
    nome="Sensor da caldeira",
    temperatura=82.7
)

print(sensor.exibir_status())

sensor.ligar()

print()
print(sensor.exibir_status())
print(f"Medição: {sensor.medir():.1f} {sensor.unidade}")

Identificador: TEMP-001
Equipamento: Sensor da caldeira
Status: desligado


Identificador: TEMP-001
Equipamento: Sensor da caldeira
Status: ligado

Medição: 82.7 °C


### O que aconteceu?

O objeto criado a partir de:

`SensorTemperatura`

Mas ee consegue utilziar métodos definidos em `Equipamento`:

```python
sensor.ligar()
sensor.desligar()
sensor.exibir_status()
```

Também possui atributos definidos sem `Sensor`:


```python
sensor.unidade
```

E possui comportamentos específicos de `SensorTemperatura`:

```python
sensor.medir()
sensor.temperatura
```


A busca ocorre pela hierarquia:

```bash
SensorTemperatura
        ↓
      Sensor
        ↓
    Equipamento
        ↓
      object
```


# 2. Substituir um método herdado da classe principal?

Substituir um método herdado é chamado **sobrescrita de método**, `method overrriding`.


Vamos criar uma classe principal para modelos de Machine Learning.

In [1]:
class ModeloML:
    def __init__(self, nome: str):
        self.nome = nome

    def descrever(self) -> str:
        return f"Modelo de Machine Learning: {self.nome}"
    
    def prever(self, dados: list[float]) -> float:
        return 0.0

Agora criamos uma classe filha que substitui o método `prever()`:

In [2]:
class ModeloFalhaEquipamento(ModeloML):
    def __init__(self, nome: str, limite: float = 0.5):
        super().__init__(nome)
        self.limite = limite

    def prever(self, dados: list[float]) -> float:
        temperatura, vibracao, horas_uso = dados

        pontuacao = (
            temperatura / 100 * 0.40
            + vibracao / 10 * 0.35
            * horas_uso / 10_000 * 0.25
        )

        return min(max(pontuacao, 0), 1)
    
    def classificar(self, dados: list[float]) -> str:
        probabilidade = self.prever(dados)

        if probabilidade >= self.limite:
            return "Possivel falha"
        
        return "Operação Normal"

### Testando

In [3]:
modelo_generico = ModeloML("Modelo Genérico")

modelo_falha = ModeloFalhaEquipamento(
    nome="Manutenção preditiva",
    limite=0.60
)

dados_sensor = [
    90.0,   #temperatura
    8.0,    #vibracao
    8500.0  #horas de uso
]

print(modelo_generico.prever(dados_sensor))

probabilidade = modelo_falha.prever(dados_sensor)

print(f"Probabilidade: {probabilidade:.2%}")
print(f"Classificação: {modelo_falha.classificar(dados_sensor)}")

0.0
Probabilidade: 41.95%
Classificação: Operação Normal


As duas classes possuem um método chamado prever(), mas cada uma apresenta um comportamento diferente.

Quando executamos:

```python
modelo_falha.prever(dados_sensor)
```


Python utiliza o método da classe `ModeloFalhaEquipamento`, porque ela sobrescreveu a implementação herdada.

### Sobrescrita usando `super()`

Também podemos aproveitar parte do método da classe principal:

In [4]:
class ModeloFalhaEquipamento(ModeloML):
    def descrever(self) -> str:
        descricao_original = super().descrever()

        return (
            f"{descricao_original}\n"
            "Finalidade: prever falhas em equipamentos industriais."
        )

In [5]:
# teste
modelo = ModeloFalhaEquipamento("Modelo de falhas")

print(modelo.descrever())

Modelo de Machine Learning: Modelo de falhas
Finalidade: prever falhas em equipamentos industriais.


Nesse caso, o método foi substituído, mas ainda utilizou a implementação original por meio de:

```python
super().descrever()
```

# 3. Criar uma classe que herda de duas classes pai

Quando uma classe herda de mais de uma classe, temos herança múltipla.

Vamos criar:

```bash
Treinavel       Persistivel
      \           /
       \         /
        ModeloPreditivo

```

### Primeira classe pai

In [7]:
class Treinavel:
    def treinar(self, dados: list[list[float]]) -> None:
        print(
            f"Treinando modelo com {len(dados)} registro."
        )

### Segunda classe pai

In [8]:
class Persistivel:
    def salvar(self, caminho: str) -> None:
        print(f"Modelo salvo em : {caminho}")

    def carregar(self, caminho: str) -> None:
        print(f"Modelo carregado de: {caminho}")

### Classe com duas classes pai

In [9]:
class ModeloPreditivo(Treinavel, Persistivel):
    def __init__(self, nome: str):
        self.nome = nome
        self.treinado = False

    def treinar(self, dados: list[list[float]]) -> None:
        super().treinar(dados)
        self.treinado = True

        print(f"O modelo '{self.nome}' foi treinado.")

    def prever(self, dados: list[float]) -> float:
        if not self.treinado:
            raise RuntimeError(
                "O modelo precisa ser treinado antes da predição."
            )
        
        return sum(dados) / len(dados)

In [10]:
# TESTANDO

modelo = ModeloPreditivo(
    nome="Modelo de manutenção preditiva"
)

dados_treinamento = [
    [70.0, 3.2, 2000],
    [85.0, 7.8, 7500],
    [65.0, 2.1, 1200]
]

modelo.treinar(dados_treinamento)

resultado = modelo.prever([82.0, 6.5, 6500])

print(f"Resultado da predição: {resultado:.2f}")

modelo.salvar("modelos/modelo_falhas.pkl")
modelo.carregar("modelos/modelo_falhas.pkl")

Treinando modelo com 3 registro.
O modelo 'Modelo de manutenção preditiva' foi treinado.
Resultado da predição: 2196.17
Modelo salvo em : modelos/modelo_falhas.pkl
Modelo carregado de: modelos/modelo_falhas.pkl


A classe `ModeloPreditivo` recebe:


* De `Treinavel`:

```python
modelo.treinar(...)
```

* De `Persistivel`:

```python
modelo.salvar(...)
modelo.carregar(...)
```

* E possui seu próprio método:

```python
modelo.prever(...)
```


Verificando a classe pai:

```python
print(ModeloPreditivo.__bases__)
```

In [11]:
print(ModeloPreditivo.__bases__)

(<class '__main__.Treinavel'>, <class '__main__.Persistivel'>)


# 4. Usar manipulação de nomes para criar uma variável de classe privada


Em Python, uma variável iniciada com dois sublinhados ativa um mecanismo chamado **name mangling**, ou manipulação de nomes.

In [13]:
class ConfiguracaoAPI:
    __limite_requisicoes = 100

    @classmethod
    def obter_limite(cls) -> int:
        return cls.__limite_requisicoes
    
    @classmethod
    def alterar_limite(cls, novo_limite: int) -> None:
        if novo_limite <= 0:
            raise ValueError(
                "O limite precisa ser maior que zero!"
            )
        
        cls.__limite_requisicoes = novo_limite

`__limite_requisicoes `é uma variável de classe porque foi criada diretamente no corpo da classe:

Ela é compartilhada por todos os objetos da classe.

#### Acessando corretamente

In [14]:
print(ConfiguracaoAPI.obter_limite())

ConfiguracaoAPI.alterar_limite(500)

print(ConfiguracaoAPI.obter_limite())

100
500


#### Tentando acessar diretamente

In [ ]:
print(ConfiguracaoAPI.__limite_requisicoes)

# Isso ocorre porque Python altera internamente o nome da variável.

AttributeError: type object 'ConfiguracaoAPI' has no attribute '__limite_requisicoes'

#### Importante

O uso de `__` não fornece segurança real.

Ele serve principalmente para:

* evitar conflitos de nomes em subclasses;
* dificultar alterações acidentais;
* indicar que o atributo é de uso interno.

Portanto:

`__limite_requisicoes`

não é completamente inacessível. Python apenas modifica o nome internamente.

#### Exemplo com contador privado de modelos

In [16]:
class ModeloRegistrado:
    __quantidade_modelos = 0

    def __init__(self, nome: str):
        self.nome = nome
        ModeloRegistrado.__quantidade_modelos += 1

    @classmethod
    def quantidade_criada(cls) -> int:
        return cls.__quantidade_modelos

In [17]:
# Teste
modelo_1 = ModeloRegistrado("Regressão logística")
modelo_2 = ModeloRegistrado("Random Forest")
modelo_3 = ModeloRegistrado("XGBoost")

print(ModeloRegistrado.quantidade_criada())

3


# 5. Ordem de resolução de métodos — MRO

### Teoria

MRO significa -> Method Resolution Order

É a ordem utilizada pelo Python para procurar atributos e métodos em uma hierarquia de classes.

Em herança simples, isso é relativamente direto:

```bash
SensorTemperatura
        ↓
    Sensor
        ↓
    Equipamento
        ↓
      object
```

Em herança múltipla, a ordem pode ser mais complexa.

Python utiliza um algoritmo chamado **linearização C3** para produzier uma ordem consistente.

Você pode consultar a MRO com:

```python
MinhaClasse.mro()
```

ou:

```python
MinhaClasse.__mro__
```

### Exemplo simples de conflito:

In [1]:
class Autenticacao:
    def validar(self) -> str:
        return "Validacao da autenticacao"

In [2]:
class BancoDados:
    def validar(self) -> str:
        return "Validacao do banco de dados"

In [3]:
# Classe filha:
class ServicoAPI(Autenticacao, BancoDados):
    pass

In [4]:
# Uso
servico = ServicoAPI()

print(servico.validar())

Validacao da autenticacao


In [5]:
for classe in ServicoAPI.mro():
    print(classe.__name__)

ServicoAPI
Autenticacao
BancoDados
object


## MRO em uma estrutura de diamante

Uma estrutura de diamante acontece quando duas classes herdam da mesma classe base e uma quarta classe herda das duas.


```bash
               ComponenteBase
                /          \
               /            \
         BancoDados      Monitoramento
               \            /
                \          /
                  PipelineML
```

Implementacao:

In [6]:
class ComponenteBase:
    def executar(self) -> list[str]:
        return ["ComponenteBase"]

In [7]:
class BancoDados(ComponenteBase):
    def executar(self) -> list[str]:
        return [
            "BancoDados"
        ] + super().executar()

In [8]:
class Monitoramento(ComponenteBase):
    def executar(self) -> list[str]:
        return [
            "Moinitoramento"
        ] + super().executar()

In [9]:
class PipelineML(BancoDados, Monitoramento):
    def executar(self) -> list[str]:
        return [
            "PipelineML"
        ] + super().executar()

In [10]:
pipeline = PipelineML()

print(pipeline.executar())

['PipelineML', 'BancoDados', 'Moinitoramento', 'ComponenteBase']


In [11]:
for classe in PipelineML.mro():
    print(classe.__name__)

PipelineML
BancoDados
Monitoramento
ComponenteBase
object
